# 04. Exploratory Data Analysis

이 노트북은 `data/processed/seoul_apt_trade_2025_features.csv`를 기준으로 모델링 전 데이터 상태와 가격 패턴을 탐색한다.

`pandas`, `numpy`, `matplotlib`만 사용해서 VS Code 커널에 `seaborn`이 없어도 실행되도록 구성했다.


## 1. 라이브러리 및 경로 설정

In [1]:
from pathlib import Path
import os
import warnings

os.environ.setdefault('MPLCONFIGDIR', str(Path('/private/tmp/matplotlib-cache')))
warnings.filterwarnings('ignore', message='Glyph.*')
warnings.filterwarnings('ignore', message='FigureCanvasAgg.*')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data/processed/seoul_apt_trade_2025_features.csv'
FIGURE_DIR = PROJECT_ROOT / 'reports/figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['axes.unicode_minus'] = False

DATA_PATH

PosixPath('/Users/cheong-kyumin/HSU/데이터마이닝/dm-apt-predict/dm-apt-predict/data/processed/seoul_apt_trade_2025_features.csv')

## 2. 데이터 로드 및 기본 구조 확인

In [2]:
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df['contract_date'] = pd.to_datetime(df['contract_date'])

df.shape

(77359, 31)

In [3]:
df.head()

,sigungu,apartment_name,area_m2,contract_ym,contract_day,price_10k_krw,floor,built_year,road_name,sido,...,longitude,distance_to_cbd_km,distance_to_ybd_km,distance_to_gbd_km,nearest_business_district_distance_km,nearest_business_district,nearest_subway_distance_km,hospital_count_within_1km,nearest_hospital_distance_km,large_mart_count_within_1km
0,서울특별시 성동구 하왕십리동,왕십리KCC스위첸,64.236,202512,31,137500,8,2016,무학봉길 35,서울특별시,...,127.026967,4.359801,10.131561,7.012161,4.359801,CBD,0.432088,1.0,0.470749,0.0
1,서울특별시 종로구 행촌동,대성아파트,94.940,202512,31,69500,3,1971,사직로 21,서울특별시,...,126.962566,1.530684,6.379895,10.112848,1.530684,CBD,0.467477,3.0,0.521502,0.0
2,서울특별시 중구 충무로4가,남산센트럴자이,82.413,202512,31,128000,18,2009,퇴계로 235,서울특별시,...,126.997824,1.802663,7.910221,7.649539,1.802663,CBD,0.352045,1.0,0.866817,0.0
3,서울특별시 성동구 마장동,현대,84.910,202512,31,118000,12,1998,살곶이길 50,서울특별시,...,127.042494,5.697082,11.785014,8.116617,5.697082,CBD,0.351577,0.0,1.151700,0.0
4,서울특별시 중구 충무로4가,남산센트럴자이,80.473,202512,31,125000,8,2009,퇴계로 235,서울특별시,...,126.997824,1.802663,7.910221,7.649539,1.802663,CBD,0.352045,1.0,0.866817,0.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 77359 entries, 0 to 77358
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   sigungu                                77359 non-null  str           
 1   apartment_name                         77359 non-null  str           
 2   area_m2                                77359 non-null  float64       
 3   contract_ym                            77359 non-null  int64         
 4   contract_day                           77359 non-null  int64         
 5   price_10k_krw                          77359 non-null  int64         
 6   floor                                  77359 non-null  int64         
 7   built_year                             77359 non-null  int64         
 8   road_name                              77359 non-null  str           
 9   sido                                   77359 non-null  str           
 1

## 3. 결측치 및 좌표 커버리지 확인

In [5]:
missing_summary = df.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).to_frame('missing_count')
missing_summary['missing_ratio'] = missing_summary['missing_count'] / len(df)
missing_summary

,missing_count,missing_ratio
matched_address,25,0.000323
latitude,25,0.000323
longitude,25,0.000323
distance_to_cbd_km,25,0.000323
distance_to_ybd_km,25,0.000323
distance_to_gbd_km,25,0.000323
nearest_business_district_distance_km,25,0.000323
nearest_business_district,25,0.000323
nearest_subway_distance_km,25,0.000323
hospital_count_within_1km,25,0.000323


In [6]:
coordinate_summary = pd.Series({
    'total_rows': len(df),
    'coordinate_rows': df[['latitude', 'longitude']].dropna().shape[0],
    'missing_coordinate_rows': df[['latitude', 'longitude']].isna().any(axis=1).sum(),
    'unique_addresses': df['full_road_address'].nunique(),
    'unique_geocoded_addresses': df.dropna(subset=['latitude', 'longitude'])['full_road_address'].nunique(),
})
coordinate_summary

total_rows                   77359
coordinate_rows              77334
missing_coordinate_rows         25
unique_addresses              5769
unique_geocoded_addresses     5762
dtype: int64

In [7]:
df.loc[df[['latitude', 'longitude']].isna().any(axis=1), [
    'apartment_name', 'gu', 'law_dong', 'full_road_address', 'geocode_status'
]].drop_duplicates().head(20)

,apartment_name,gu,law_dong,full_road_address,geocode_status
2030,홍제역해링턴플레이스,서대문구,홍제동,서울특별시 서대문구,NaN
7613,"수정가,나동",은평구,수색동,서울특별시 은평구 은평터널로2길 21,not_found
9405,롯데캐슬리버파크시그니쳐,광진구,자양동,서울특별시 광진구,NaN
11525,태릉해링턴플레이스,노원구,공릉동,서울특별시 노원구,NaN
24667,서대문센트럴아이파크,서대문구,홍은동,서울특별시 서대문구,NaN
44267,새마을,마포구,연남동,서울특별시 마포구 성미산로23길 54,not_found
61701,해링턴플레이스안암,성북구,안암동3가,서울특별시 성북구,NaN
71380,센트럴파크,용산구,한강로3가,서울특별시 용산구,NaN


## 4. 주요 수치 변수 기초 통계

In [8]:
numeric_columns = [
    'price_10k_krw', 'price_per_m2_10k_krw', 'area_m2', 'floor', 'built_year', 'age',
    'nearest_subway_distance_km', 'nearest_business_district_distance_km',
    'hospital_count_within_1km', 'nearest_hospital_distance_km', 'large_mart_count_within_1km',
]
df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
price_10k_krw,77359.0,126914.471762,94830.780763,6500.000000,70500.000000,104500.000000,154000.000000,2.900000e+06
price_per_m2_10k_krw,77359.0,1640.967786,890.164494,181.582361,1033.525018,1412.835136,1971.253730,1.058672e+04
area_m2,77359.0,76.074963,27.793934,11.330000,59.785000,81.750000,84.960000,3.173600e+02
floor,77359.0,9.712496,6.479812,-2.000000,5.000000,9.000000,13.000000,6.700000e+01
built_year,77359.0,2003.318528,11.402045,1961.000000,1996.000000,2003.000000,2012.000000,2.025000e+03
age,77359.0,21.681472,11.402045,0.000000,13.000000,22.000000,29.000000,6.400000e+01
nearest_subway_distance_km,77334.0,0.523211,0.305797,0.017757,0.317248,0.462052,0.644645,3.124931e+00
nearest_business_district_distance_km,77334.0,6.424270,3.301941,0.169217,3.868028,5.770691,8.667571,1.547732e+01
hospital_count_within_1km,77334.0,0.419402,0.647328,0.000000,0.000000,0.000000,1.000000,3.000000e+00
nearest_hospital_distance_km,77334.0,1.382965,0.734675,0.031689,0.814303,1.295397,1.851100,4.651939e+00


In [9]:
df['log_price_10k_krw'] = np.log1p(df['price_10k_krw'])
df['log_price_per_m2_10k_krw'] = np.log1p(df['price_per_m2_10k_krw'])
df[['price_10k_krw', 'log_price_10k_krw', 'price_per_m2_10k_krw', 'log_price_per_m2_10k_krw']].describe().T

,count,mean,std,min,25%,50%,75%,max
price_10k_krw,77359.0,126914.471762,94830.780763,6500.000000,70500.000000,104500.000000,154000.000000,2.900000e+06
log_price_10k_krw,77359.0,11.544978,0.647762,8.779711,11.163382,11.556952,11.944714,1.488022e+01
price_per_m2_10k_krw,77359.0,1640.967786,890.164494,181.582361,1033.525018,1412.835136,1971.253730,1.058672e+04
log_price_per_m2_10k_krw,77359.0,7.285139,0.473831,5.207201,6.941698,7.254061,7.586932,9.267450e+00


## 5. 가격 분포 확인

In [10]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
plots = [
    ('price_10k_krw', 'Price distribution', 'Price (10k KRW)'),
    ('log_price_10k_krw', 'Log price distribution', 'log1p(price)'),
    ('price_per_m2_10k_krw', 'Price per m2 distribution', 'Price per m2 (10k KRW)'),
    ('log_price_per_m2_10k_krw', 'Log price per m2 distribution', 'log1p(price per m2)'),
]
for ax, (column, title, xlabel) in zip(axes.ravel(), plots):
    ax.hist(df[column].dropna(), bins=60, color='#4C78A8', alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')
plt.tight_layout()
fig.savefig(FIGURE_DIR / 'eda_price_distributions.png', dpi=100, bbox_inches='tight')
plt.close(fig)

In [11]:
df[['price_10k_krw', 'price_per_m2_10k_krw']].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

,price_10k_krw,price_per_m2_10k_krw
0.01,15000.0,554.992170
0.05,36000.0,712.843056
0.25,70500.0,1033.525018
0.50,104500.0,1412.835136
0.75,154000.0,1971.253730
0.95,299000.0,3449.113153
0.99,470000.0,4911.205969


## 6. 자치구별 거래량 및 가격

In [12]:
gu_summary = (
    df.groupby('gu')
    .agg(
        transaction_count=('price_10k_krw', 'size'),
        mean_price_10k=('price_10k_krw', 'mean'),
        median_price_10k=('price_10k_krw', 'median'),
        mean_price_per_m2=('price_per_m2_10k_krw', 'mean'),
        median_price_per_m2=('price_per_m2_10k_krw', 'median'),
        mean_area_m2=('area_m2', 'mean'),
    )
    .sort_values('mean_price_per_m2', ascending=False)
)
gu_summary

,transaction_count,mean_price_10k,median_price_10k,mean_price_per_m2,median_price_per_m2,mean_area_m2
gu,,,,,,
강남구,3912,280564.133691,250000.0,3194.149997,3090.304109,87.228372
서초구,2941,268215.411765,242000.0,2909.035184,2500.416736,94.497092
용산구,1444,209915.047784,173000.0,2334.889809,2124.144442,87.707224
송파구,5326,186585.565340,169000.0,2301.309232,2048.263685,83.569236
성동구,4227,159148.866572,148000.0,2065.500805,1947.821981,78.261421
마포구,3988,147125.571715,141000.0,1947.179291,1871.926208,77.384590
광진구,2011,142405.148682,139000.0,1774.331932,1778.144136,80.306868
양천구,3401,134892.869744,124000.0,1719.056257,1506.059536,79.714442
강동구,5019,125245.679817,125000.0,1681.384213,1610.392720,75.818928


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

count_by_gu = df['gu'].value_counts().sort_values()
axes[0].barh(count_by_gu.index, count_by_gu.values, color='#4C78A8')
axes[0].set_title('Transactions by gu')
axes[0].set_xlabel('Transaction count')

price_by_gu = gu_summary['mean_price_per_m2'].sort_values()
axes[1].barh(price_by_gu.index, price_by_gu.values, color='#F58518')
axes[1].set_title('Mean price per m2 by gu')
axes[1].set_xlabel('Mean price per m2')

plt.tight_layout()
fig.savefig(FIGURE_DIR / 'eda_gu_volume_price.png', dpi=100, bbox_inches='tight')
plt.close(fig)

In [14]:
box_data = [df.loc[df['gu'].eq(gu), 'price_per_m2_10k_krw'].dropna().values for gu in gu_summary.index]
fig, ax = plt.subplots(figsize=(16, 7))
ax.boxplot(box_data, labels=gu_summary.index, showfliers=False)
ax.set_title('Price per m2 by gu')
ax.set_xlabel('gu')
ax.set_ylabel('Price per m2 (10k KRW)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
fig.savefig(FIGURE_DIR / 'eda_gu_price_boxplot.png', dpi=100, bbox_inches='tight')
plt.close(fig)

/var/folders/dn/hj27qly116330wnl76ppktb40000gn/T/ipykernel_83929/578661800.py:3: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(box_data, labels=gu_summary.index, showfliers=False)


## 7. 거래 시점별 가격 추이

In [15]:
monthly_summary = (
    df.groupby('contract_month')
    .agg(
        transaction_count=('price_10k_krw', 'size'),
        mean_price_per_m2=('price_per_m2_10k_krw', 'mean'),
        median_price_per_m2=('price_per_m2_10k_krw', 'median'),
    )
    .reset_index()
)
monthly_summary

,contract_month,transaction_count,mean_price_per_m2,median_price_per_m2
0,1,3345,1639.730739,1397.306397
1,2,6363,1844.711632,1569.338502
2,3,9797,1785.230993,1551.866243
3,4,5227,1472.883950,1309.211676
4,5,7576,1567.727476,1382.237420
5,6,11266,1663.821267,1445.202730
6,7,4148,1624.231186,1300.758785
7,8,4279,1443.254246,1271.785210
8,9,8668,1624.697926,1452.931965
9,10,8535,1645.216267,1414.760670


In [16]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(monthly_summary['contract_month'], monthly_summary['mean_price_per_m2'], marker='o', color='#F58518')
ax1.set_title('Monthly price per m2 and volume')
ax1.set_xlabel('Month')
ax1.set_ylabel('Mean price per m2')
ax2 = ax1.twinx()
ax2.bar(monthly_summary['contract_month'], monthly_summary['transaction_count'], alpha=0.25, color='#4C78A8')
ax2.set_ylabel('Transaction count')
plt.tight_layout()
fig.savefig(FIGURE_DIR / 'eda_monthly_price_volume.png', dpi=100, bbox_inches='tight')
plt.close(fig)

## 8. 면적, 연식, 층과 가격 관계

In [17]:
relationship_vars = ['area_m2', 'age', 'floor']
sample = df.sample(min(5000, len(df)), random_state=42)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, column in zip(axes, relationship_vars):
    ax.scatter(sample[column], sample['price_per_m2_10k_krw'], alpha=0.25, s=12, color='#4C78A8')
    ax.set_title(f'{column} vs price per m2')
    ax.set_xlabel(column)
    ax.set_ylabel('Price per m2 (10k KRW)')
plt.tight_layout()
fig.savefig(FIGURE_DIR / 'eda_core_numeric_relationships.png', dpi=100, bbox_inches='tight')
plt.close(fig)

In [18]:
df['area_group'] = pd.cut(df['area_m2'], bins=[0, 40, 60, 85, 135, np.inf], labels=['<=40', '40-60', '60-85', '85-135', '>135'])
df['age_group'] = pd.cut(df['age'], bins=[-1, 5, 10, 20, 30, np.inf], labels=['<=5', '6-10', '11-20', '21-30', '>30'])
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, group_col, title in [(axes[0], 'area_group', 'Price per m2 by area group'), (axes[1], 'age_group', 'Price per m2 by age group')]:
    categories = df[group_col].cat.categories
    data = [df.loc[df[group_col].eq(cat), 'price_per_m2_10k_krw'].dropna().values for cat in categories]
    ax.boxplot(data, labels=categories, showfliers=False)
    ax.set_title(title)
    ax.set_xlabel(group_col)
    ax.set_ylabel('Price per m2 (10k KRW)')
plt.tight_layout()
fig.savefig(FIGURE_DIR / 'eda_area_age_groups.png', dpi=100, bbox_inches='tight')
plt.close(fig)

/var/folders/dn/hj27qly116330wnl76ppktb40000gn/T/ipykernel_83929/1518077083.py:7: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=categories, showfliers=False)
/var/folders/dn/hj27qly116330wnl76ppktb40000gn/T/ipykernel_83929/1518077083.py:7: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=categories, showfliers=False)


## 9. 입지 접근성 변수와 가격 관계

In [19]:
accessibility_cols = [
    'nearest_subway_distance_km', 'nearest_business_district_distance_km',
    'nearest_hospital_distance_km', 'hospital_count_within_1km', 'large_mart_count_within_1km',
]
df[accessibility_cols + ['price_per_m2_10k_krw']].describe().T

,count,mean,std,min,25%,50%,75%,max
nearest_subway_distance_km,77334.0,0.523211,0.305797,0.017757,0.317248,0.462052,0.644645,3.124931
nearest_business_district_distance_km,77334.0,6.424270,3.301941,0.169217,3.868028,5.770691,8.667571,15.477320
nearest_hospital_distance_km,77334.0,1.382965,0.734675,0.031689,0.814303,1.295397,1.851100,4.651939
hospital_count_within_1km,77334.0,0.419402,0.647328,0.000000,0.000000,0.000000,1.000000,3.000000
large_mart_count_within_1km,77334.0,0.561306,1.033924,0.000000,0.000000,0.000000,1.000000,7.000000
price_per_m2_10k_krw,77359.0,1640.967786,890.164494,181.582361,1033.525018,1412.835136,1971.253730,10586.723519


In [20]:
coord_df = df.dropna(subset=['latitude', 'longitude']).sample(min(5000, df[['latitude', 'longitude']].dropna().shape[0]), random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()
for ax, column in zip(axes, accessibility_cols):
    ax.scatter(coord_df[column], coord_df['price_per_m2_10k_krw'], alpha=0.2, s=12, color='#4C78A8')
    ax.set_title(f'{column} vs price per m2')
    ax.set_xlabel(column)
    ax.set_ylabel('Price per m2 (10k KRW)')
axes[-1].axis('off')
plt.tight_layout()
fig.savefig(FIGURE_DIR / 'eda_accessibility_relationships.png', dpi=100, bbox_inches='tight')
plt.close(fig)

In [21]:
district_order = ['CBD', 'YBD', 'GBD']
box_data = [df.loc[df['nearest_business_district'].eq(d), 'price_per_m2_10k_krw'].dropna().values for d in district_order]
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(box_data, labels=district_order, showfliers=False)
ax.set_title('Price per m2 by nearest business district')
ax.set_xlabel('Nearest business district')
ax.set_ylabel('Price per m2 (10k KRW)')
plt.tight_layout()
fig.savefig(FIGURE_DIR / 'eda_business_district_boxplot.png', dpi=100, bbox_inches='tight')
plt.close(fig)

/var/folders/dn/hj27qly116330wnl76ppktb40000gn/T/ipykernel_83929/12621184.py:4: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(box_data, labels=district_order, showfliers=False)


## 10. 상관관계 확인

In [22]:
correlation_columns = [
    'price_per_m2_10k_krw', 'price_10k_krw', 'area_m2', 'floor', 'age',
    'nearest_subway_distance_km', 'nearest_business_district_distance_km',
    'nearest_hospital_distance_km', 'hospital_count_within_1km', 'large_mart_count_within_1km',
]
corr = df[correlation_columns].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(11, 8))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax)
ax.set_title('Numeric feature correlation')
plt.tight_layout()
fig.savefig(FIGURE_DIR / 'eda_correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.close(fig)
corr['price_per_m2_10k_krw'].sort_values(ascending=False)

price_per_m2_10k_krw                     1.000000
price_10k_krw                            0.804112
floor                                    0.154099
area_m2                                  0.083987
large_mart_count_within_1km             -0.005805
nearest_hospital_distance_km            -0.021284
hospital_count_within_1km               -0.030751
age                                     -0.111690
nearest_subway_distance_km              -0.168347
nearest_business_district_distance_km   -0.341438
Name: price_per_m2_10k_krw, dtype: float64

## 11. 모델링 전 정리

EDA 결과를 바탕으로 모델링 단계에서 다음 사항을 검토한다.

- 타깃 후보: `price_per_m2_10k_krw`를 우선 사용하고, 필요하면 `log_price_per_m2_10k_krw`도 비교
- 기본 설명 변수: 면적, 층, 연식, 거래월, 자치구, 법정동
- 입지 변수: 지하철 거리, 업무지구 거리, 병원 거리/개수, 대형마트 개수
- 범주형 변수: `gu`, `law_dong`, `nearest_business_district` 인코딩 필요
- 결측 처리: 좌표 없는 25행은 제거하거나 입지 변수 결측 행으로 별도 처리
- 이상치 처리: 가격 상하위 1%를 확인한 뒤 제거 여부 결정


In [23]:
modeling_candidate_columns = [
    'price_per_m2_10k_krw', 'log_price_per_m2_10k_krw', 'area_m2', 'floor', 'age',
    'contract_month', 'gu', 'law_dong', 'nearest_subway_distance_km',
    'distance_to_cbd_km', 'distance_to_ybd_km', 'distance_to_gbd_km',
    'nearest_business_district_distance_km', 'nearest_business_district',
    'hospital_count_within_1km', 'nearest_hospital_distance_km', 'large_mart_count_within_1km',
]
pd.DataFrame({
    'column': modeling_candidate_columns,
    'exists': [column in df.columns for column in modeling_candidate_columns],
    'missing_count': [df[column].isna().sum() if column in df.columns else None for column in modeling_candidate_columns],
})

,column,exists,missing_count
0,price_per_m2_10k_krw,True,0
1,log_price_per_m2_10k_krw,True,0
2,area_m2,True,0
3,floor,True,0
4,age,True,0
5,contract_month,True,0
6,gu,True,0
7,law_dong,True,0
8,nearest_subway_distance_km,True,25
9,distance_to_cbd_km,True,25
